## 1. Импорт библиотек и загрузка данных

**Цель ноутбука:** Обучить baseline-модели для 4 задач бинарной классификации.

**Задачи:**
1. IC50 > медианы — соединение активно
2. CC50 > медианы — соединение малотоксично
3. SI > медианы — соединение селективно
4. SI > 8 — соединение высокоселективно

**Загружаемые данные:**
- Масштабированные признаки (X_train, X_test)
- Бинарные метки для 4 задач (y_train_clf, y_test_clf)

In [2]:
import sys
import os
import pandas as pd
import numpy as np
import joblib

sys.path.append(os.path.dirname(os.getcwd()))

from src import get_classification_models
from src import evaluate_classification

In [3]:
SAVE_PATH = '../data/processed/'

X_train = joblib.load(f'{SAVE_PATH}X_train_scaled.pkl')
X_test = joblib.load(f'{SAVE_PATH}X_test_scaled.pkl')
y_train_clf = joblib.load(f'{SAVE_PATH}y_train_clf.pkl')
y_test_clf = joblib.load(f'{SAVE_PATH}y_test_clf.pkl')

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Целевые переменные: {y_train_clf.columns.tolist()}")

X_train: (798, 165), X_test: (200, 165)
Целевые переменные: ['IC50_binary', 'CC50_binary', 'SI_binary', 'SI_8_binary']


## 2. Обучение baseline-моделей

Используются 5 моделей с параметрами по умолчанию:
- Logistic Regression (линейный классификатор)
- Decision Tree (дерево решений)
- Random Forest (ансамбль деревьев)
- Gradient Boosting (градиентный бустинг)
- XGBoost (оптимизированный градиентный бустинг)

**Метрики оценки:**
- Accuracy — доля правильных ответов
- F1 — гармоническое среднее Precision и Recall (основная метрика)
- AUC — площадь под ROC-кривой (способность разделять классы)

In [4]:
CLASSIFICATION_TASKS = {
    'IC50_binary': 'IC50 > медианы',
    'CC50_binary': 'CC50 > медианы',
    'SI_binary': 'SI > медианы',
    'SI_8_binary': 'SI > 8'
}

models = get_classification_models()

# Структура для результатов
results = {task: {} for task in CLASSIFICATION_TASKS.keys()}

## 3. Цикл обучения и оценки

Для каждой задачи:
1. Извлекаем соответствующий таргет (y_train, y_test)
2. Обучаем каждую модель на тренировочных данных
3. Делаем предсказания на тестовых данных
4. Вычисляем метрики качества (Accuracy, F1, AUC)
5. Сохраняем результаты в словарь

**Важно:** Для AUC используются вероятности классов (predict_proba), а не бинарные предсказания.

In [5]:
for task_col, task_name in CLASSIFICATION_TASKS.items():
    print(f"ЗАДАЧА: {task_name} ({task_col})")
    print('─'*60)
    
    y_train = y_train_clf[task_col]
    y_test = y_test_clf[task_col]
    
    for name, model in models.items():
        # Обучение
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Вероятности для AUC
        y_proba = None
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test)
        
        # Оценка
        metrics = evaluate_classification(y_test, y_pred, y_proba)
        results[task_col][name] = metrics
        
        # Вывод
        print(f"{name:20} | Accuracy: {metrics['Accuracy']:.4f} | F1: {metrics['F1']:.4f} | AUC: {metrics.get('AUC', 0):.4f}")


ЗАДАЧА: IC50 > медианы (IC50_binary)
────────────────────────────────────────────────────────────
LogisticRegression   | Accuracy: 0.6850 | F1: 0.6837 | AUC: 0.7518
DecisionTree         | Accuracy: 0.6800 | F1: 0.6801 | AUC: 0.7170
RandomForest         | Accuracy: 0.7100 | F1: 0.7101 | AUC: 0.7867
GradientBoosting     | Accuracy: 0.7250 | F1: 0.7251 | AUC: 0.7666
XGBoost              | Accuracy: 0.7000 | F1: 0.7000 | AUC: 0.7722
ЗАДАЧА: CC50 > медианы (CC50_binary)
────────────────────────────────────────────────────────────
LogisticRegression   | Accuracy: 0.7350 | F1: 0.7351 | AUC: 0.8389
DecisionTree         | Accuracy: 0.7050 | F1: 0.7039 | AUC: 0.7346
RandomForest         | Accuracy: 0.7800 | F1: 0.7801 | AUC: 0.8575
GradientBoosting     | Accuracy: 0.7800 | F1: 0.7802 | AUC: 0.8476
XGBoost              | Accuracy: 0.7850 | F1: 0.7852 | AUC: 0.8679
ЗАДАЧА: SI > медианы (SI_binary)
────────────────────────────────────────────────────────────
LogisticRegression   | Accuracy: 0.6150 

In [6]:
for task_col, task_name in CLASSIFICATION_TASKS.items():
    print(f"{task_name} — Результаты")
    print('─'*60)
    
    df = pd.DataFrame(results[task_col]).T
    display(df.round(4))
    
    # Сохраняем
    df.to_csv(f'../artifacts/results/classification_baseline_{task_col}.csv')

IC50 > медианы — Результаты
────────────────────────────────────────────────────────────


,Accuracy,Precision,Recall,F1,AUC
LogisticRegression,0.685,0.6949,0.685,0.6837,0.7518
DecisionTree,0.680,0.6804,0.680,0.6801,0.7170
RandomForest,0.710,0.7104,0.710,0.7101,0.7867
GradientBoosting,0.725,0.7256,0.725,0.7251,0.7666
XGBoost,0.700,0.7000,0.700,0.7000,0.7722


CC50 > медианы — Результаты
────────────────────────────────────────────────────────────


,Accuracy,Precision,Recall,F1,AUC
LogisticRegression,0.735,0.7352,0.735,0.7351,0.8389
DecisionTree,0.705,0.7186,0.705,0.7039,0.7346
RandomForest,0.780,0.7859,0.780,0.7801,0.8575
GradientBoosting,0.780,0.7811,0.780,0.7802,0.8476
XGBoost,0.785,0.7901,0.785,0.7852,0.8679


SI > медианы — Результаты
────────────────────────────────────────────────────────────


,Accuracy,Precision,Recall,F1,AUC
LogisticRegression,0.615,0.6156,0.615,0.6145,0.6642
DecisionTree,0.660,0.6603,0.660,0.6599,0.6620
RandomForest,0.670,0.6701,0.670,0.6700,0.7162
GradientBoosting,0.645,0.6451,0.645,0.6449,0.7037
XGBoost,0.640,0.6401,0.640,0.6400,0.6989


SI > 8 — Результаты
────────────────────────────────────────────────────────────


,Accuracy,Precision,Recall,F1,AUC
LogisticRegression,0.685,0.6765,0.685,0.6767,0.6948
DecisionTree,0.685,0.6827,0.685,0.6837,0.6826
RandomForest,0.710,0.7035,0.710,0.6973,0.7478
GradientBoosting,0.730,0.7247,0.730,0.7235,0.7457
XGBoost,0.730,0.7247,0.730,0.7235,0.7636


In [7]:
summary_f1 = pd.DataFrame({
    'IC50 > медианы': [results['IC50_binary'][name]['F1'] for name in models.keys()],
    'CC50 > медианы': [results['CC50_binary'][name]['F1'] for name in models.keys()],
    'SI > медианы': [results['SI_binary'][name]['F1'] for name in models.keys()],
    'SI > 8': [results['SI_8_binary'][name]['F1'] for name in models.keys()]
}, index=models.keys())

display(summary_f1.round(4))

# Сохраняем
summary_f1.to_csv('../artifacts/results/classification_baseline_summary.csv')

,IC50 > медианы,CC50 > медианы,SI > медианы,SI > 8
LogisticRegression,0.6837,0.7351,0.6145,0.6767
DecisionTree,0.6801,0.7039,0.6599,0.6837
RandomForest,0.7101,0.7801,0.6700,0.6973
GradientBoosting,0.7251,0.7802,0.6449,0.7235
XGBoost,0.7000,0.7852,0.6400,0.7235


## Выбор топ-моделей для оптимизации

Для каждой задачи выбираем по 2 модели с наивысшим F1.



In [8]:
top_models = {}

for task_col, task_name in CLASSIFICATION_TASKS.items():
    # Сортируем по F1
    sorted_models = sorted(
        results[task_col].items(),
        key=lambda x: x[1]['F1'],
        reverse=True
    )
    top_2 = [name for name, _ in sorted_models[:2]]
    top_models[task_col] = top_2
    
    print(f"\n{task_name}:")
    for i, name in enumerate(top_2, 1):
        print(f"  {i}. {name} (F1 = {results[task_col][name]['F1']:.4f})")

# Сохраняем топ-модели
joblib.dump(top_models, '../artifacts/top_models_classification.pkl')


IC50 > медианы:
  1. GradientBoosting (F1 = 0.7251)
  2. RandomForest (F1 = 0.7101)

CC50 > медианы:
  1. XGBoost (F1 = 0.7852)
  2. GradientBoosting (F1 = 0.7802)

SI > медианы:
  1. RandomForest (F1 = 0.6700)
  2. DecisionTree (F1 = 0.6599)

SI > 8:
  1. GradientBoosting (F1 = 0.7235)
  2. XGBoost (F1 = 0.7235)


['../artifacts/top_models_classification.pkl']

**Результат:**
- **IC50 > медианы**: GradientBoosting (0.73) и RandomForest (0.71)
- **CC50 > медианы**: XGBoost (0.79) и GradientBoosting (0.78)
- **SI > медианы**: RandomForest (0.67) и DecisionTree (0.66)
- **SI > 8**: GradientBoosting (0.72) и XGBoost (0.72)

Выбранные модели будут оптимизированы в ноутбуке `06_classification_tuning.ipynb`.